# DeepCT (2019)
[[paper]](https://arxiv.org/pdf/1910.10687)<br>
DeepCT = Deep Contextualized Term Weighting

__DeepCT__ — это метод <u>обучаемого</u> разреженного информационного поиска (Learned Sparse Retrieval), основанный замене стандартной частоты слова (TF) на вычисляемую нейросетью величину, который позволяет повысить точность первого этапа поиска (First Stage Retrieval) за счет учета семантики и сохранить его быстроту за счет возможности использования инвертированного индекса

__Постановка задачи__<br>
Решается классическая задача инфомрационного поиска. По заданному запросу $Q$ найти $K$ наиболее релевантных документов из коллекции $D = \{d_1, d_2, \ldots, d_N\}$

__Мотивация__<br>
Традиционный Sparse Retrieval поиск (BM25) полагается на статистики Term Frequency (TF) — чем чаще слово встречается в тексте, тем оно важнее. В реальности же вес слова при мэтчинге сильно зависит от окружающего слово контекста в конкретном документе. Для нормального учета контекста был предложен более современный подход Dense Retrieval, основанный на вычислении эмбедингов Трансформерными модделями. Иго минусом однако является то, что он менее точный и требует поддержания approximate индекса

__Существующие подходы__<br>
На момент выхода модели существовали следующие подходы:
- BM25: классический Sparse Retrieval - частоты без контекста не учитывают семантику
- Deep Relevance Matching Models (DRMM, 2016): используют эмбеддинги для сопоставления запроса и документа, но слишком медленны для поиска по всей коллекции
- BERT re-rankers (2019): кросс-энкодерная архитекутура (early interaction) - хорошо учитывает контекст, но они могут применяться только к малому подмножеству документов (этап re-ranking), так как требуют прогона всех пар "запрос-документ" через трансформер
- Doc2query (2019): генерирует возможные вопросы к документу и добавляет их в текст. Это расширяет документ (Expansion), но не исправляет веса уже существующих в нем слов

__<u>Идея</u>__<br>
Давайте использовать Sparse Retrieval, но вместо вычисления эвристик TF параметризуем их через нейросетевую модель. В результате получим семантически "богатое" представление каждого слова с высокой скоростью поиска характерной для Sparse Retrieval

__Архитектура__<br>
DeepCT использует стандартный BERT (обычно `base-uncased`) в качестве энкодера
1.  Input = документ
2.  Encoder - BERT генерирует контекстуализированные векторы для каждого токена.
3.  Output Layer: поверх последнего слоя BERT добавляется полносвязный слой (Linear Layer) с одним выходом. Для каждого токена $i$ предсказывается скалярное значение $w_i$, которое отражает его важность.
4.  Aggregation: Поскольку BERT работает с WordPiece-токенами, веса для подслов агрегируются (например, берется вес первого подслова), чтобы получить итоговый вес для целого слова.

__Алгоритм обучения__<br>
обучаем на задаче предсказания Query-Term Frequency (QTF)
1.  берем большой датасет с парами, например, MS MARCO
2.  делаем self-supervised разметку: для каждого слова $t$ из документа $D$ и всех релевантных ему запросов $\{q_1, q_2, ...\}$ считаем частоту слова $t$ из документа
3.  решаем задачу регрессии. Минимизируем MSE между предсказанным весом $w_i$ и нормализованным значением QTF
    $$\mathcal{L} = \sum (w_{predicted} - \text{normalized\_QTF})^2$$

__Алгоритм инференса__<br>
Процесс применения DeepCT происходит **offline** (до начала поиска):
1.  Обработка коллекции: Каждый документ в базе пропускается через обученный DeepCT
2.  Трансформация весов: Полученные дробные веса $w_i$ масштабируются в целые числа (например, путем умножения на константу и округления). Это делается для того, чтобы "обмануть" поисковый движок.
3.  Индексация: В инвертированный индекс (Lucene/ElasticSearch) вместо реального количества упоминаний слова (TF) записывается предсказанное значение. Например, если слово "apple" встречается в тексте 1 раз, но DeepCT считает его важным, в индекс записывается, что оно встретилось там 15 раз.
4.  Retrieval: Поиск выполняется стандартным алгоритмом BM25. Поскольку веса в индексе уже контекстуализированы, BM25 автоматически выдает более качественные результаты без дополнительных затрат времени при обработке запроса.

__Результаты__<br>
Сравнивали на датасете MS MARCO (Passage Retrieval):
- Recall@1000 дает +10пп против классического BM25. Метрика MRR@10 +30%
- сохранили скорость ответа
- DeepCT первым предложил использовать BERT не для сравнения запроса с документом, а для предобработки (enrichment) статического индекса, сохраняя при этом эффективность классического поиска

## Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import torch.nn.functional as F

# Define the DeepCT model
class DeepCT(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased'):
        super(DeepCT, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.linear = nn.Linear(self.bert.config.hidden_size, 1)  # Linear layer to predict term importance

    def forward(self, input_ids, attention_mask):
        # Get contextualized embeddings from BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Use the last hidden state
        last_hidden_state = outputs.last_hidden_state
        # Predict term importance for each token
        term_importance = self.linear(last_hidden_state).squeeze(-1)
        return term_importance

# Initialize tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = DeepCT()

# Example passage
passage = "The condition of the patient was stable, but the condition of the contract was under review."

# Tokenize the passage
inputs = tokenizer(passage, return_tensors='pt', truncation=True, padding=True)

# Forward pass through the model
with torch.no_grad():
    term_importance_scores = model(inputs['input_ids'], inputs['attention_mask'])

# Convert token IDs back to words
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# Aggregate scores for subwords to get word-level importance
word_importance = {}
for token, score in zip(tokens, term_importance_scores[0]):
    if token.startswith("##"):
        # Append score to the previous word
        word_importance[last_word] += score.item()
    else:
        # Start a new word
        last_word = token
        word_importance[last_word] = score.item()

# Display the importance scores
print("Word Importance Scores:")
for word, score in word_importance.items():
    print(f"{word}: {score:.4f}")

# Note: In a real-world scenario, you would train the model using a dataset like MS MARCO
# and use the Query-Term Frequency (QTF) as labels for supervised learning.
# Here, we demonstrate the inference step with a pre-trained BERT model and random weights.
```

### Explanation:

1. **Model Architecture**: 
   - The `DeepCT` class extends `nn.Module` and uses a pre-trained BERT model as the encoder.
   - A linear layer is added on top of BERT to predict the importance of each token.

2. **Tokenization**:
   - The passage is tokenized using BERT's tokenizer, which handles subword tokenization.

3. **Inference**:
   - The model predicts a scalar importance score for each token in the passage.
   - Scores for subword tokens are aggregated to get word-level importance.

4. **Output**:
   - The script prints the importance scores for each word in the passage.

This example illustrates how DeepCT uses BERT to predict term importance based on context, replacing traditional term frequency with context-aware weights. In practice, you would train this model with labeled data to learn meaningful importance scores.